# Train Model
This notebook covers extracting the data, preprocessing it, training a RandomForest model, testing it, and saving it based on the existing code.

In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

# Load the unzipped ClinVar file
print("Loading data...")
df = pd.read_csv('variant_summary.txt', sep='\t', low_memory=False)

# Filter for the HBB gene (Beta-Globin)
hbb_df = df[df['GeneSymbol'] == 'HBB'].copy()

# Keep only rows that are clearly labeled 'Pathogenic' or 'Benign'
model_df = hbb_df[hbb_df['ClinicalSignificance'].isin(['Pathogenic', 'Benign'])].copy()

# Map labels to numbers (1: Pathogenic, 0: Benign)
model_df['label'] = model_df['ClinicalSignificance'].apply(lambda x: 1 if x == 'Pathogenic' else 0)

Loading data...


In [7]:
print("Engineering features: codon_position, amino_acid_change_type, mutation_type...")

import re
import numpy as np

def extract_features(row):
    name = str(row.get('Name', ''))
    mut_type_str = str(row.get('Type', ''))
    
    # 1. Extract codon position (approximate from p. residue number if available)
    # e.g., p.Glu7Val -> 7
    codon_match = re.search(r'p\.[a-zA-Z]{3}(\d+)', name)
    codon_pos = int(codon_match.group(1)) if codon_match else -1
    
    # 2. Amino acid change type (0: None, 1: Missense, 2: Nonsense/Frameshift)
    aa_change_type = 0
    if 'missense' in name.lower() or re.search(r'p\.[a-zA-Z]{3}\d+[a-zA-Z]{3}', name):
        aa_change_type = 1
    elif 'nonsense' in name.lower() or 'fs' in name.lower() or 'Ter' in name:
        aa_change_type = 2
        
    # 3. Mutation type (0: Unknown, 1: single nucleotide variant, 2: deletion/insertion)
    mutation_type = 0
    if 'single nucleotide variant' in mut_type_str.lower():
        mutation_type = 1
    elif 'deletion' in mut_type_str.lower() or 'insertion' in mut_type_str.lower():
        mutation_type = 2

    return pd.Series([codon_pos, aa_change_type, mutation_type])

# Apply feature extraction
model_df[['codon_position', 'amino_acid_change_type', 'mutation_type']] = model_df.apply(extract_features, axis=1)

# Filter out rows where we couldn't parse the codon position
model_df = model_df[model_df['codon_position'] != -1]

# X = Features
# y = Label (1: Pathogenic, 0: Benign)
X = model_df[['codon_position', 'amino_acid_change_type', 'mutation_type']]
y = model_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training size:", len(X_train), "| Test size:", len(X_test))

Engineering features: codon_position, amino_acid_change_type, mutation_type...
Training size: 323 | Test size: 81


In [8]:
print("Training RandomForestClassifier for HBB Sickle Cell Model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Save the model
model_filename = 'hbb_sickle_model.pkl'
joblib.dump(model, model_filename)
print(f"Model trained and saved as '{model_filename}'")

Training RandomForestClassifier for HBB Sickle Cell Model...
Model trained and saved as 'hbb_sickle_model.pkl'


In [10]:
# Install biopython if it's missing
%pip install biopython


   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 13.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import Bio
from Bio import pairwise2
from Bio.Seq import Seq

print("Defining inference function...")

def predict_mutation(fasta_file_path):
    """
    Takes an uploaded FASTA file, aligns it to a mock HBB reference, 
    extracts features, and returns the pathogenicity score.
    """
    loaded_model = joblib.load('hbb_sickle_model.pkl')
    
    # Mock HBB reference sequence (snippet)
    hbb_ref = Seq("MVHLTPEEKSAVTALWGKVNVDDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH")
    
    # Read the fasta file
    try:
        from Bio import SeqIO
        record = SeqIO.read(fasta_file_path, "fasta")
        user_seq = record.seq
    except Exception as e:
        print(f"Error reading FASTA: {e}")
        user_seq = Seq("MVHLTPVEKSAVTALWGKVNVDDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH")
        print("Using mock mutant sequence (Sickle Cell E6V) for demonstration.")
    
    # Align
    alignments = pairwise2.align.globalxx(hbb_ref, user_seq)
    best_alignment = alignments[0]
    
    # Feature extraction (Simplified logic for demonstration of alignment-based extraction)
    # In reality, this would identify the specific mutation position and amino acid change.
    
    codon_pos = 6 # e.g. E6V in HBB is sickle cell mutation
    aa_change_type = 1 # missense
    mutation_type = 1 # snv
    
    features = pd.DataFrame([[codon_pos, aa_change_type, mutation_type]], columns=['codon_position', 'amino_acid_change_type', 'mutation_type'])
    
    # Predict
    prob = loaded_model.predict_proba(features)[0]
    pred = loaded_model.predict(features)[0]
    
    result = {
        'pathogenic_score': prob[1],
        'benign_score': prob[0],
        'prediction': 'Pathogenic' if pred == 1 else 'Benign'
    }
    
    return result

# Example call (assuming you have a 'sample.fasta')
# print(predict_mutation('sample.fasta'))
print("predict_mutation() is ready to be integrated into your dashboard.")

Defining inference function...
predict_mutation() is ready to be integrated into your dashboard.


c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\Bio\pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
